## Perbaikan v3.1 — zoom dan caption

**SPEC / DESIGN:** karakteristik teks memakai caption yang membedakan estimasi rata-rata/CI bootstrap dari uji Wilcoxon–Holm. ETHICS menambahkan pembesaran jumlah kata 5× dengan penanda CI/titik keluar rentang. Heatmap dan judul Man/Woman tetap. Gambar pronomina historis dipertahankan.

Jalankan hanya tiga sel kode berikut. Hasil memakai statistik v2 dan referensi heatmap v3, lalu ditulis ke run baru. Rencana/verifikasi: docs/gender_lm_zoom_caption_20261007.md. Cetak fisik dan integrasi naskah belum dilakukan.


In [1]:
Z_BENCHMARK="GSM8K"
# Additive correction: reuse frozen v3 inputs and preserve all previous assets.
from pathlib import Path
import ast, json
import nbformat
Z_ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p/"AGENTS.md").exists())
Z_PATH = Z_ROOT/"notebooks/gender/linguistic marker"/Z_BENCHMARK/("linguistic_marker_gender_gsm8k_direct_gender_v3.ipynb" if Z_BENCHMARK=="GSM8K" else "ETHICS_Gender_Linguistic_Markers_Analysis_v1_5_candidate.ipynb")
Z_NOTEBOOK = nbformat.read(Z_PATH,as_version=4)
z_setup = next(c.source for c in Z_NOTEBOOK.cells if c.id=="gender-bab6-v3-1")
assert z_setup.count('bab6_v3_')==2
exec(compile(z_setup.replace("bab6_v3_","bab6_v3_1_"),str(Z_PATH)+"::v31-inputs","exec"))
Z_V3_ID = "20261007_230650_742192" if Z_BENCHMARK=="GSM8K" else "20261007_230725_835481"
Z_V3_RUN = CP_ROOT/"result/analyses/linguistic_markers/runs"/f"gender_{Z_BENCHMARK.lower()}_bab6_v3_{Z_V3_ID}"
Z_V3_INDEX = pd.read_csv(Z_V3_RUN/"figure_index.csv")
Z_V3_MANIFEST = json.loads((Z_V3_RUN/"manifest.json").read_text(encoding="utf-8"))
Z_PRIOR_HASHES={str(p):cp_hash(p) for p in Z_V3_RUN.rglob("*") if p.is_file()}
Z_V3_ACTIVE_HASHES={str(p):cp_hash(p) for p in Path(Z_V3_MANIFEST["active_figures"]).iterdir() if p.is_file()}
Z_PRONOUN_FILES = [
    CP_ROOT/"notebooks/gender/img/12_gender_pronoun_prevalence_by_model_persona.png",
    CP_ROOT/"notebooks/gender/img/13_gender_pronoun_direct_man_vs_woman_forest.png",
    CP_ROOT/"notebooks/gender/linguistic marker/candidate/figures/gsm8k_07_pronoun_profile.png",
    CP_ROOT/"notebooks/gender/linguistic marker/candidate/figures/gsm8k_08_pronoun_direct.png",
]
Z_PRONOUN_HASHES={str(p):cp_hash(p) for p in Z_PRONOUN_FILES}
Z_ZOOM_AUDIT=[]
z_render_source=next(c.source for c in Z_NOTEBOOK.cells if c.id=="gender-bab6-v3-2")
z_tree=ast.parse(z_render_source)
for z_node in z_tree.body:
    if isinstance(z_node,ast.FunctionDef) or (isinstance(z_node,ast.Assign) and any(isinstance(t,ast.Name) and t.id in {"CP_CHAR_LABELS","CP_CHAR_SHORT"} for t in z_node.targets)):
        z_segment=ast.get_source_segment(z_render_source,z_node)
        if isinstance(z_node,ast.FunctionDef) and z_node.name=="cp_characteristic":
            z_segment=z_segment.replace("Titik: selisih rata-rata · garis: CI 95% · k/n: berubah/tersedia",
                "Titik: selisih rata-rata · garis: CI bootstrap 95% rata-rata")
            z_segment=z_segment.replace("* p Holm < 0,05 · segitiga: n<20 · CI ditampilkan jika n≥2",
                "* Wilcoxon–Holm p<0,05 (bukan uji rata-rata) · CI jika n≥2")
            z_segment=z_segment.replace("TP: tak berubah · ND: tak tersedia · positif: sisi kiri lebih tinggi",
                "k/n: berubah/tersedia · segitiga: n<20 · TP/ND: tak berubah/tak tersedia")
            z_segment=z_segment.replace("I1–I3: variasi warmup · Kandidat · skala sama antarperbandingan",
                "Positif: kiri lebih tinggi · I1–I3: variasi warmup · Kandidat")
            z_segment=z_segment.replace('proof=scope in ["gsm8k","commonsense"] and role=="man_vs_neutral" and marker=="word_count"',"proof=False")
        exec(compile(z_segment,str(Z_PATH)+"::v31-reused-renderer","exec"))
Z_BASE_CAPTION=cp_caption
def cp_caption(scope,role_title,title,kind):
    caption=Z_BASE_CAPTION(scope,role_title,title,kind)
    if kind in ("characteristic","word_count_zoom"):
        caption += (" Titik dan CI mengestimasi selisih rata-rata. Bintang berasal dari uji "
            "Wilcoxon signed-rank dua sisi setelah koreksi Holm pada keluarga v2 yang sama; "
            "uji ini menggunakan peringkat selisih berpasangan, bukan pengujian selisih rata-rata "
            "yang diestimasi CI bootstrap. Karena itu CI yang melintasi nol tidak otomatis "
            "bertentangan dengan bintang. P untuk pasangan tanpa perubahan mengikuti konvensi sumber.")
    return caption
CP_PROOF = PdfPages(CP_RUN/"pratinjau_A4_caption_zoom.pdf")
CP_PROOF_COUNT=0
CP_PROOF_FILES=[]
print("V3.1",CP_BENCHMARK,CP_RUN,flush=True)


GSM8K: 2,268 perbandingan v2; 16 penanda inti; run 20261007_234710_219408
V3.1 GSM8K D:\TugasAkhir\result\analyses\linguistic_markers\runs\gender_gsm8k_bab6_v3_1_20261007_234710_219408


In [2]:
def z_print_page(fig,title):
    global CP_PROOF_COUNT
    width,height=fig.get_size_inches()*2.54
    old_axes=[(ax,ax.get_position().frozen()) for ax in fig.axes]
    old_text=[(tx,tx.get_position()) for tx in fig.texts]
    fig.set_size_inches(21*CP_CM,29.7*CP_CM)
    sx,sy,ox,oy=14/21,height/29.7,4/21,(29.7-3-height)/29.7
    for ax,p in old_axes:
        ax.set_position([ox+p.x0*sx,oy+p.y0*sy,p.width*sx,p.height*sy])
    for tx,(x,y) in old_text:
        tx.set_position((ox+x*sx,oy+y*sy))
    caption=("Titik menunjukkan selisih rata-rata jumlah kata dan garis menunjukkan CI bootstrap "
        "95% marginal untuk selisih rata-rata. Bintang berasal dari uji Wilcoxon signed-rank "
        f"dua sisi setelah koreksi Holm ({CP_FAMILY_SIZE} hipotesis/model), bukan pengujian "
        "selisih rata-rata yang sama dengan CI. k/n: pasangan berubah/tersedia. ")
    if CP_BENCHMARK=="ETHICS":
        caption+=("Kiri: rentang lengkap; kanan: pembesaran 5× sekitar nol. Ujung panah menunjukkan "
            "CI di luar rentang; segitiga terbuka menunjukkan titik di luar rentang. ")
    caption+="I1–I3 adalah variasi warmup. Status pengukuran: kandidat."
    added=fig.text(4/21,oy-.012,textwrap.fill(caption,width=89),ha="left",va="top",fontsize=8,linespacing=1.25)
    rect=Rectangle((4/21,3/29.7),14/21,23.7/29.7,transform=fig.transFigure,fill=False,
        edgecolor="#CBD5E1",linewidth=.5,linestyle="--")
    fig.add_artist(rect)
    fig.canvas.draw()
    box=added.get_window_extent(fig.canvas.get_renderer())
    assert box.y0 >= (3/29.7)*fig.bbox.height, "Caption exceeds thesis text block"
    assert box.x1 <= (18/21)*fig.bbox.width, "Caption exceeds 14 cm text width"
    CP_PROOF.savefig(fig,facecolor="white")
    name=f"pratinjau_A4_caption_{CP_PROOF_COUNT+1:02d}.png"
    fig.savefig(CP_RUN/name,dpi=120,facecolor="white")
    CP_PROOF_FILES.append(dict(file=name,title=title,width_cm=14))
    CP_PROOF_COUNT+=1
    added.remove();rect.remove()
    fig.set_size_inches(width*CP_CM,height*CP_CM)
    for ax,p in old_axes: ax.set_position(p)
    for tx,pos in old_text: tx.set_position(pos)

Z_FINISH_BASE=cp_finish
def cp_finish(fig,stem,scope,role,kind,group,title,caption,nrows=0,proof=False):
    Z_FINISH_BASE(fig,stem,scope,role,kind,group,title,caption,nrows,False)
    selected=(CP_BENCHMARK=="ETHICS" and scope=="commonsense" and role=="direct" and kind=="word_count_zoom") or (
        CP_BENCHMARK=="GSM8K" and role=="man_vs_neutral" and stem.endswith("__00_word_count"))
    if selected:
        z_print_page(fig,title)

def z_word_count_zoom(scope,role,role_title,conditions):
    part=CP_DATA[CP_DATA.subset.eq(scope)&CP_DATA.role.eq(role)&CP_DATA.marker_id.eq("word_count")]
    all_part=CP_DATA[CP_DATA.subset.eq(scope)&CP_DATA.marker_id.eq("word_count")]
    vals=all_part[["effect","ci_display_low","ci_display_high"]].to_numpy().ravel()
    vals=vals[np.isfinite(vals)]
    full_bound=max(max(abs(vals))*1.12,.001) if len(vals) else 1.12
    zoom_bound=full_bound/5
    rows=len(conditions)
    height=19.5 if rows==4 else 16.5
    fig,axes=plt.subplots(rows,2,figsize=(14*CP_CM,height*CP_CM),squeeze=False)
    fig.subplots_adjust(left=.205,right=.89,top=.885,bottom=.175,wspace=.6,hspace=.60)
    for i,condition in enumerate(conditions):
        selected=part[part.condition.eq(condition)].set_index("model_id")
        for j,bound in enumerate([full_bound,zoom_bound]):
            ax=axes[i,j]
            for y,model in enumerate(CP_MODELS):
                r=selected.loc[model]
                n,k=int(r.n_pairs),int(r.n_changed)
                lo,hi,point=r.ci_display_low,r.ci_display_high,r.effect
                low_cut=bool(np.isfinite(lo) and lo < -bound)
                high_cut=bool(np.isfinite(hi) and hi > bound)
                point_cut=bool(np.isfinite(point) and abs(point)>bound)
                if np.isfinite(lo):
                    if max(lo,-bound)<=min(hi,bound):
                        ax.plot([max(lo,-bound),min(hi,bound)],[y,y],color=CP_COLORS[model],lw=1.0,
                            ls="--" if n<20 else "-")
                    if low_cut:
                        ax.scatter(-bound,y,marker="<",color=CP_COLORS[model],s=18,clip_on=False,zorder=4)
                    if high_cut:
                        ax.scatter(bound,y,marker=">",color=CP_COLORS[model],s=18,clip_on=False,zorder=4)
                if np.isfinite(point):
                    if point_cut:
                        ax.scatter(np.sign(point)*bound,y+.14,marker=">" if point>0 else "<",
                            facecolors="white",edgecolors=CP_COLORS[model],s=30,clip_on=False,zorder=5)
                    else:
                        ax.scatter(point,y,marker="^" if n<20 else "o",color=CP_COLORS[model],s=15,zorder=3)
                    if r.significant_display:
                        ax.annotate("*",(np.clip(point,-bound,bound),y),xytext=(0,3),
                            textcoords="offset points",fontsize=7,ha="center")
                else:
                    ax.text(0,y,"ND",fontsize=7,ha="center",va="center")
                if j==1:
                    ax.text(1.035,y,f"{k}/{n}"+(" TP" if n>0 and k==0 else ""),transform=ax.get_yaxis_transform(),fontsize=7,va="center")
                    Z_ZOOM_AUDIT.append(dict(source_row_id=int(r.source_row_id),subset=scope,role=role,
                        condition=condition,model_id=model,n_pairs=n,n_changed=k,
                        point=point,ci_low=lo,ci_high=hi,full_low=-full_bound,full_high=full_bound,
                        zoom_low=-bound,zoom_high=bound,ci_lower_arrow=low_cut,ci_upper_arrow=high_cut,
                        point_outside_marker=point_cut))
                else:
                    assert not low_cut and not high_cut and not point_cut
            ax.set_xlim(-bound,bound);ax.set_ylim(5.6,-.6)
            ax.set_yticks(range(6),[CP_NAMES[m] for m in CP_MODELS] if j==0 else [""]*6,fontsize=7.2)
            suffix="−N" if role.endswith("neutral") else "−E" if role.endswith("explicit") else ""
            ax.set_title(CP_SHORT[condition]+suffix+" · "+("Lengkap" if j==0 else "Pembesaran 5×"),fontsize=8,fontweight="bold",pad=5)
            if j==1: ax.text(1.035,1.035,"k/n",transform=ax.transAxes,fontsize=7)
            ax.xaxis.set_major_locator(MaxNLocator(nbins=3));ax.tick_params(axis="x",labelsize=7)
            ax.axvline(0,color="#94A3B8",lw=.6);ax.grid(axis="x",color="#E2E8F0",lw=.5)
            ax.spines[["top","right"]].set_visible(False)
    title=f"{cp_scope(scope)} · {role_title}"
    fig.suptitle(title+"\nJumlah kata: lengkap dan pembesaran",fontsize=10,fontweight="bold",y=.98)
    fig.text(.5,.126,"Selisih rata-rata jumlah kata (kata)",ha="center",fontsize=8.5)
    fig.text(.025,.095,"Titik: selisih rata-rata · garis: CI bootstrap 95% rata-rata",fontsize=7)
    fig.text(.025,.073,"* Wilcoxon–Holm p<0,05: uji peringkat, bukan uji rata-rata",fontsize=7)
    fig.text(.025,.051,"Ujung panah: CI keluar rentang · segitiga terbuka: titik di luar",fontsize=7)
    fig.text(.025,.029,"k/n: berubah/tersedia · ▲: n<20 · CI jika n≥2 · TP/ND",fontsize=7)
    fig.text(.025,.008,"Positif: kiri lebih tinggi · I1–I3: variasi warmup · Kandidat",fontsize=7)
    caption=cp_caption(scope,role_title,"Selisih rata-rata jumlah kata","characteristic")+(
        " Panel kiri mempertahankan seluruh titik/CI; panel kanan memakai rentang seperlima panel "
        "kiri (pembesaran 5× sekitar nol), seragam untuk semua model dan kontras pada subset. "
        "Ujung panah pada garis menandai CI melampaui batas; segitiga terbuka di tepi menandai "
        "titik di luar rentang, bukan estimasi pada batas. Nilai sebenarnya tetap ada pada panel "
        "lengkap dan tabel. Semua model dipertahankan tanpa seleksi signifikansi.")
    stem=f"{scope}__{CP_ROLES.index((role,role_title,conditions))+1:02d}_{role}__00_word_count_zoom"
    cp_finish(fig,stem,scope,role,"word_count_zoom","00_karakteristik",title+" · Jumlah kata lengkap/zoom",caption)

for z_scope in CP_SCOPES:
    for z_role,z_title,z_conditions in CP_ROLES:
        for z_marker in CP_CHARACTER:
            cp_characteristic(z_scope,z_role,z_title,z_conditions,z_marker)
        if CP_BENCHMARK=="ETHICS":
            z_word_count_zoom(z_scope,z_role,z_title,z_conditions)
CP_PROOF.close()
CP_INDEX=pd.DataFrame(CP_INDEX)
CP_INDEX.to_csv(CP_RUN/"figure_index.csv",index=False,encoding="utf-8-sig")
pd.DataFrame(Z_ZOOM_AUDIT).to_csv(CP_RUN/"zoom_clipping_audit.csv",index=False,encoding="utf-8-sig")
print("Render V3.1 selesai",len(CP_INDEX),"PNG/PDF",flush=True)


GSM8K: 10 gambar PNG + PDF selesai
Render V3.1 selesai 15 PNG/PDF


In [3]:
expected=len(CP_SCOPES)*len(CP_ROLES)*(4 if CP_BENCHMARK=="ETHICS" else 3)
assert len(CP_INDEX)==expected
assert sorted(CP_CHAR_ROWS)==sorted(CP_DATA.loc[CP_DATA.marker_id.isin(CP_CHARACTER),"source_row_id"])
roundtrip=pd.read_csv(CP_RUN/"all_comparisons_native_units.csv",float_precision="round_trip",low_memory=False)
pd.testing.assert_frame_equal(CP_DATA,roundtrip[CP_DATA.columns],check_dtype=False)
if CP_BENCHMARK=="ETHICS":
    za=pd.DataFrame(Z_ZOOM_AUDIT)
    assert len(za)==len(CP_DATA[CP_DATA.marker_id.eq("word_count")])
    assert np.allclose(za.full_high/za.zoom_high,5)
    assert (za.ci_lower_arrow==(za.ci_low<za.zoom_low)).all()
    assert (za.ci_upper_arrow==(za.ci_high>za.zoom_high)).all()
    assert (za.point_outside_marker==(za.point.abs()>za.zoom_high)).all()
    assert za.ci_lower_arrow.any() or za.ci_upper_arrow.any()
assert all(cp_hash(p)==h for p,h in CP_INPUT_HASHES.items())
assert all(cp_hash(p)==h for p,h in Z_PRIOR_HASHES.items())
assert all(cp_hash(p)==h for p,h in Z_V3_ACTIVE_HASHES.items())
assert all(cp_hash(p)==h for p,h in Z_PRONOUN_HASHES.items())
for r in CP_INDEX.itertuples():
    with Image.open(CP_STAGE/r.figure) as im: im.load()
    assert b"/Subtype /Image" not in (CP_STAGE/r.pdf).read_bytes()
    assert cp_hash(CP_STAGE/r.figure)==r.png_sha256 and cp_hash(CP_STAGE/r.pdf)==r.pdf_sha256
assert not CP_DEST.exists()
shutil.copytree(CP_STAGE,CP_DEST)
for r in CP_INDEX.itertuples():
    assert cp_hash(CP_DEST/r.figure)==r.png_sha256 and cp_hash(CP_DEST/r.pdf)==r.pdf_sha256
CP_INDEX["png_path"]=CP_INDEX.figure.map(lambda x:str(CP_DEST/x))
CP_INDEX["pdf_path"]=CP_INDEX.pdf.map(lambda x:str(CP_DEST/x))
inherited=Z_V3_INDEX[Z_V3_INDEX.kind.ne("characteristic")].copy()
inherited["png_path"]=inherited.figure.map(lambda x:str(Path(Z_V3_MANIFEST["active_figures"])/x))
inherited["pdf_path"]=inherited.pdf.map(lambda x:str(Path(Z_V3_MANIFEST["active_figures"])/x))
for r in inherited.itertuples():
    assert cp_hash(r.png_path)==r.png_sha256 and cp_hash(r.pdf_path)==r.pdf_sha256
combined=pd.concat([inherited,CP_INDEX],ignore_index=True)
assert len(combined)==len(Z_V3_INDEX)+(25 if CP_BENCHMARK=="ETHICS" else 0)
combined.to_csv(CP_RUN/"combined_figure_index.csv",index=False,encoding="utf-8-sig")
combined.to_csv(CP_DEST/"combined_figure_index.csv",index=False,encoding="utf-8-sig")
pd.DataFrame([dict(path=p,sha256=h,status="historical_support_not_v3_measurement",
    source_guide="notebooks/gender/README_LINGUISTIC_MARKER.md") for p,h in Z_PRONOUN_HASHES.items()]).to_csv(
    CP_RUN/"preserved_pronoun_figures.csv",index=False,encoding="utf-8-sig")
manifest=dict(run=str(CP_RUN),benchmark=CP_BENCHMARK,version="bab6_v3.1_zoom_caption",
    source_v2=str(CP_SOURCE),source_v3=str(Z_V3_RUN),new_figures=len(CP_INDEX),inherited_figures=len(inherited),
    combined_figures=len(combined),active_figures=str(CP_DEST),comparisons=len(CP_DATA),
    statistics_unchanged=True,heatmaps_unchanged=True,pronoun_files_preserved=Z_PRONOUN_HASHES,
    source_hashes=CP_INPUT_HASHES,previous_v3_hashes=Z_PRIOR_HASHES,previous_active_hashes=Z_V3_ACTIVE_HASHES,
    zoom_factor=5,ci_lower_arrows=int(za.ci_lower_arrow.sum()) if CP_BENCHMARK=="ETHICS" else 0,
    ci_upper_arrows=int(za.ci_upper_arrow.sum()) if CP_BENCHMARK=="ETHICS" else 0,
    points_outside_zoom=int(za.point_outside_marker.sum()) if CP_BENCHMARK=="ETHICS" else 0,
    proof_pdf=str(CP_RUN/"pratinjau_A4_caption_zoom.pdf"),proof_pages=CP_PROOF_COUNT,
    width_cm=14,maximum_height_cm=float(CP_INDEX.height_cm.max()),
    caption_semantics="bootstrap_CI_of_mean_difference_vs_Wilcoxon_Holm_inference",
    physical_print_checked=False,thesis_files_modified=False,scientific_status="candidate",
    notebook=str(Z_PATH),notebook_sha256=cp_hash(Z_PATH))
(CP_RUN/"manifest.json").write_text(json.dumps(manifest,ensure_ascii=False,indent=2),encoding="utf-8")
snapshot=nbformat.v4.new_notebook(cells=[c for c in Z_NOTEBOOK.cells if c.get("metadata",{}).get("presentation_section")=="gender_bab6_v31"],metadata=Z_NOTEBOOK.metadata)
nbformat.validate(snapshot);nbformat.write(snapshot,CP_RUN/"presentation_notebook.ipynb")
lines=[f"# {CP_BENCHMARK} gender — perbaikan caption dan zoom v3.1","",
    f"{len(CP_INDEX)} gambar baru, {len(inherited)} heatmap/cakupan v3 tetap dipakai. Judul Man/Woman dipertahankan.",
    "", "Titik dan CI mengestimasi selisih rata-rata. Bintang berasal dari Wilcoxon signed-rank dua sisi "
    f"dengan Holm ({CP_FAMILY_SIZE} hipotesis/model). Bintang bukan keputusan uji rata-rata dari CI bootstrap. "
    "CI adalah interval marginal 95%; CI melintasi nol tidak otomatis bertentangan dengan bintang.",
    "", "Grafik lengkap tetap tersedia. ETHICS menambahkan panel lengkap dan pembesaran 5× untuk jumlah kata. "
    "Panah horizontal pada CI menunjukkan interval melampaui batas zoom. Segitiga terbuka kiri/kanan "
    "menunjukkan estimasi titik di luar rentang; posisi penanda di tepi bukan nilai estimasi. "
    "Segitiga menghadap atas tetap menandai n<20. Semua model disajikan pada kedua panel.",
    "", "Buka pratinjau_A4_caption_zoom.pdf pada ukuran sebenarnya atau cetak 100%. "
    "Halaman ETHICS menampilkan Commonsense Man−Woman: empat skenario, enam model, lengkap/zoom, "
    "dan caption yang membedakan estimasi rata-rata dari uji Wilcoxon. Inspeksi layar tidak menggantikan cetak fisik.",
    "", "combined_figure_index.csv memuat tautan seluruh paket yang dipakai dan caption terbaru. "
    "Grafik karakteristik lama tetap ada sebagai versi historis; gunakan caption v3.1 untuk integrasi. "
    "zoom_clipping_audit.csv mencatat setiap batas, nilai sebenarnya, dan penanda keluar rentang. "
    "Seluruh angka tetap dari tabel v2: all_comparisons_native_units.csv.",
    "", "preserved_pronoun_figures.csv menunjuk empat file pronomina lama dengan hash. "
    "Dua gambar utama di notebooks/gender/img dan dua tampilan candidate tetap tersedia. "
    "Sumber historisnya berbeda dari rebuild v3; jangan menyebutnya hasil pengukuran v3. "
    "Notebook pendukung: notebooks/gender/linguistic marker/pendukung/linguistic_marker_gender_words_pronouns.ipynb.",
    "", "Jalankan hanya tiga sel kode bagian **Perbaikan v3.1 — zoom dan caption** pada notebook aktif. "
    "Hasil ditulis ke folder baru setiap kali. Naskah belum disinkronkan. Status kamus tetap kandidat.",
    "", "## Caption karakteristik dan zoom"]
for r in CP_INDEX.itertuples():
    lines += ["",f"### {r.figure}","",r.caption]
readme="\n".join(lines)+"\n"
(CP_RUN/"README.md").write_text(readme,encoding="utf-8")
(CP_DEST/"README.md").write_text(readme,encoding="utf-8")
(CP_ACTIVE/f"PRESENTASI_BAB6_V31_{CP_ID}.json").write_text(json.dumps(
    dict(run=str(CP_RUN),figures=str(CP_DEST),combined_index=str(CP_RUN/"combined_figure_index.csv")),
    ensure_ascii=False,indent=2),encoding="utf-8")
print("SELESAI V3.1",CP_BENCHMARK,CP_RUN,flush=True)
print("FIGURES",CP_DEST,flush=True)
print("PROOF",CP_RUN/"pratinjau_A4_caption_zoom.pdf",flush=True)


SELESAI V3.1 GSM8K D:\TugasAkhir\result\analyses\linguistic_markers\runs\gender_gsm8k_bab6_v3_1_20261007_234710_219408
FIGURES D:\TugasAkhir\notebooks\gender\linguistic marker\GSM8K\figures\bab6_v3_1_20261007_234710_219408
PROOF D:\TugasAkhir\result\analyses\linguistic_markers\runs\gender_gsm8k_bab6_v3_1_20261007_234710_219408\pratinjau_A4_caption_zoom.pdf
